In [1]:
import pandas as pd
import numpy as np
from unidecode import unidecode
import re
from rapidfuzz import fuzz, process

In [2]:
df = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\police-shootings-export.csv")
df2 = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\Full_MedianHouseHoldIncome2015-2023.csv"  ,encoding='latin-1')
df3 = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\FULL_PeopleBelowPovertyLevel2015-2023.csv" ,encoding='latin-1')
df4 = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\Full_PrecentOfPeopleOver25CompletedHighSchool2015-2023.csv",encoding='latin-1' , low_memory=False )
df5 = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\Full_Race2015-2023.csv",encoding='latin-1')
States = pd.read_csv(r"D:\CRIME DATA SET\DATA\Project\data\States_in_America.csv",encoding='latin-1')

In [3]:
df2 = pd.merge(df2 , df3 , on=['year' , 'City' , 'State'] , how='left')
df2 = pd.merge(df2 , df4 , on=['year' , 'City' , 'State'] , how='left')
df2 = pd.merge(df2 , df5 , on=['year' , 'City' , 'State'] , how='left')

# Cleaning Fact Table

#

# Cleaning Date Column 

In [4]:
df['date'] = pd.to_datetime(df['date'] , format = '%Y-%m-%d')

In [5]:
df['date'].isnull().sum()

0

In [6]:
df['year'] = df['date'].dt.year

# Cleaning Name Column

In [7]:
df['name'].isnull().sum()

337

In [8]:
df['name'].fillna('unknown' , inplace=True)

In [9]:
df['name'].isnull().sum()

0

# Cleaning age Column

In [10]:
df['age'].isnull().sum()

384

In [11]:
df['age'].mean()

37.40950678304764

In [12]:
df['age'].median()

35.0

In [13]:
df['age'].mode()

0    33.0
Name: age, dtype: float64

In [14]:
df['age'].fillna(df['age'].median() , inplace=True)

In [15]:
df['age'].sort_values().head(10)

7237     2.0
9463     4.0
2906     6.0
834      6.0
6640     8.0
7165    12.0
1015    12.0
7456    13.0
1681    13.0
9840    13.0
Name: age, dtype: float64

In [16]:
df['age'] =df['age'].astype(int)

In [17]:
Q1=df['age'].quantile(0.25)
Q2=df['age'].quantile(0.5)
Q3=df['age'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + IQR * 1.5
lower= Q1 - IQR * 1.5

df[(df['age'] > upper)  | (df['age'] < lower)]
##this data is not a mistake its a real accedent happend with childrens and adults

,date,name,age,gender,armed,race,city,state,flee,body_camera,signs_of_mental_illness,police_departments_involved,year
20,2015-01-09,Jimmy Foreman,71,male,gun,White,England,AR,not,False,False,"England Police Department, AR",2015
28,2015-01-14,Talbot Schroeder,75,male,knife,White,Old Bridge,NJ,not,False,False,"Old Bridge Police Department, NJ",2015
91,2015-02-07,James Allen,74,male,gun,Black,Gastonia,NC,not,False,False,"Gastonia Police Department, NC",2015
125,2015-02-20,Douglas Harris,77,male,gun,Black,Birmingham,AL,not,False,True,"Birmingham Police Department, AL",2015
275,2015-04-12,Richard Dale Weaver,83,male,knife,White,Oklahoma City,OK,not,False,True,"Oklahoma City Police Department, OK",2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9308,2024-01-22,Janet Sours,81,female,knife,White,Wildwood,FL,not,False,True,"Sumter County Sheriff's Department, FL",2024
9387,2024-02-18,Wendall Cross,85,male,gun,White,Blairsville,GA,not,False,False,"Union County Sheriff's Office, GA",2024
9678,2024-05-14,Aubrey James Osteen,77,male,gun,Unknown,Tucumcari,NM,not,False,False,"Quay County Sheriff's Office, NM",2024
9694,2024-05-17,Eugene Mewes,80,male,gun,Unknown,East Moline,IL,NaN,False,False,"East Moline Police Department, IL",2024


# Cleaning gender Column

In [18]:
df['gender'].isnull().sum()

28

In [19]:
(df['gender'] == 'male').sum()/len(df)*100

95.2289497624583

In [20]:
df['gender'].fillna(df['gender'].mode()[0] , inplace=True)

In [21]:
(df['gender'] == 'male').sum()/len(df)*100

95.51197816638027

# Cleaning armed Column

In [22]:
for tool in df['armed'].unique() :
    print(tool)

gun
unarmed
other
replica
knife
blunt_object
nan
vehicle
undetermined
other,gun
unknown
blunt_object,blunt_object
gun,knife
knife,blunt_object
vehicle,gun
gun,vehicle
replica,vehicle
blunt_object,knife
knife,vehicle
vehicle,knife,other
knife,knife
replica,knife
other,blunt_object,knife
other,knife
vehicle,knife
gun,other
blunt_object,other


In [23]:
#  undetermined = unknown = nan
#  blunt_object,blunt_object = blunt_object
#  knife,knife = knife
#  knife,vehicle = vehicle,knife
#  other,gun = gun,other
#  blunt_object,knife = knife,blunt_object

In [24]:
round(df['armed'].isnull().sum()/len(df)*100 , 2)

2.13

In [25]:
df['armed'].replace(['undetermined' , 'unknown' ] , np.nan , inplace=True)

In [26]:
df['armed'].isnull().sum()/len(df)*100

7.884362680683312

In [27]:
df.groupby('armed')['armed'].count().sort_values(ascending = False)

armed
gun                          5748
knife                        1683
unarmed                       551
vehicle                       351
replica                       314
blunt_object                  239
other                          98
gun,knife                      40
gun,vehicle                    38
vehicle,gun                    20
knife,knife                     5
knife,blunt_object              4
other,gun                       4
knife,vehicle                   3
blunt_object,blunt_object       3
blunt_object,knife              3
vehicle,knife                   2
blunt_object,other              1
other,knife                     1
replica,vehicle                 1
replica,knife                   1
gun,other                       1
other,blunt_object,knife        1
vehicle,knife,other             1
Name: armed, dtype: int64

In [28]:
df['armed'].replace('vehicle,gun' , 'gun,vehicle' ,  inplace=True)
df['armed'].replace('vehicle,knife' , 'knife,vehicle' ,  inplace=True)
df['armed'].replace('blunt_object,knife' , 'knife,blunt_object' ,  inplace=True)
df['armed'].replace('blunt_object,blunt_object' , 'blunt_object' ,  inplace=True)
df['armed'].replace('knife,knife' , 'knife' ,  inplace=True)
df['armed'] = df['armed'].str.replace(',' , ' & ')

In [29]:
df.groupby('armed')['armed'].count().sort_values(ascending = False)

armed
gun                             5748
knife                           1688
unarmed                          551
vehicle                          351
replica                          314
blunt_object                     242
other                             98
gun & vehicle                     58
gun & knife                       40
knife & blunt_object               7
knife & vehicle                    5
other & gun                        4
blunt_object & other               1
other & blunt_object & knife       1
other & knife                      1
replica & knife                    1
replica & vehicle                  1
gun & other                        1
vehicle & knife & other            1
Name: armed, dtype: int64

In [30]:
df['armed'].isnull().sum()/len(df)*100

7.884362680683312

In [31]:
distribution =df['armed'].value_counts(normalize=True)

In [32]:
distribution

armed
gun                             0.630747
knife                           0.185230
unarmed                         0.060463
vehicle                         0.038516
replica                         0.034456
blunt_object                    0.026555
other                           0.010754
gun & vehicle                   0.006365
gun & knife                     0.004389
knife & blunt_object            0.000768
knife & vehicle                 0.000549
other & gun                     0.000439
replica & vehicle               0.000110
vehicle & knife & other         0.000110
replica & knife                 0.000110
other & blunt_object & knife    0.000110
other & knife                   0.000110
gun & other                     0.000110
blunt_object & other            0.000110
Name: proportion, dtype: float64

In [33]:
Number_of_nulls = df['armed'].isnull().sum()

In [34]:

# Fill missing 'armed' values by categories based on their original distribution

if  Number_of_nulls> 0:   
    fill_values = np.random.choice(distribution.index , size = Number_of_nulls , p= distribution.values)
    df.loc[df['armed'].isnull(), 'armed'] = fill_values

In [35]:
df['armed'].isnull().sum()

0

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9893 entries, 0 to 9892
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         9893 non-null   datetime64[ns]
 1   name                         9893 non-null   object        
 2   age                          9893 non-null   int32         
 3   gender                       9893 non-null   object        
 4   armed                        9893 non-null   object        
 5   race                         9893 non-null   object        
 6   city                         9821 non-null   object        
 7   state                        9893 non-null   object        
 8   flee                         8548 non-null   object        
 9   body_camera                  9893 non-null   bool          
 10  signs_of_mental_illness      9893 non-null   bool          
 11  police_departments_involved  9892 non-null 

# Cleaning race column

In [37]:
x=0
for i in df['race'].unique() :
    x+=1
    print(x , i )

1 White
2 Asian
3 Hispanic
4 Black
5 Other
6 Unknown
7 Native American
8 White,Black,Native American
9 Native American,Hispanic
10 White,Hispanic
11 Black,Hispanic
12 White,Black


In [38]:
df['race'].value_counts()

race
White                          4432
Black                          2346
Hispanic                       1623
Unknown                        1144
Asian                           175
Native American                 135
Other                            31
Black,Hispanic                    2
White,Black                       2
White,Black,Native American       1
Native American,Hispanic          1
White,Hispanic                    1
Name: count, dtype: int64

# Cleaning City Columns

In [39]:
df['city'].isnull().sum()

72

In [40]:
cities  = df[df['city'].isnull()]
cities

,date,name,age,gender,armed,race,city,state,flee,body_camera,signs_of_mental_illness,police_departments_involved,year
2197,2017-03-20,Clarence Duane Huderle,73,male,gun,Unknown,NaN,MN,NaN,False,False,"Polk County Sheriff's Office, MN",2017
8005,2022-12-02,Jonathan Wiseman,39,male,gun,White,NaN,DE,other,False,False,"Delaware State Police, DE",2022
8027,2022-12-09,Michael Fredric Stevens,57,male,gun,White,NaN,VA,not,False,False,"Rockbridge County Sheriff's Office, VA",2022
8048,2022-12-17,unknown,35,male,gun,Unknown,NaN,WV,NaN,False,False,"McDowell County Sheriff's Office, WV",2022
8054,2022-12-19,Michael Cline,35,male,knife,White,NaN,VA,foot,False,False,"Louisa County Sheriff's Office, VA",2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9197,2023-12-16,unknown,35,male,gun,Unknown,NaN,VA,NaN,False,False,"Stafford County Sheriff's Department, VA",2023
9211,2023-12-20,David Estrada,38,male,gun,Unknown,NaN,CO,NaN,False,False,"Adams County Sheriff's Department, CO",2023
9225,2023-12-24,Efren Inda,50,male,gun,Hispanic,NaN,CO,NaN,False,False,"Boulder County Sheriff's Office, CO;Colorado S...",2023
9595,2024-04-27,Brian James Propes,42,male,gun,White,NaN,FL,not,False,False,"Washington County Sheriff's Office, FL",2024


In [41]:
sorted = df[['city' , 'state' , 'police_departments_involved']].sort_values(by =['police_departments_involved'])
sorted

,city,state,police_departments_involved
6423,Abbeville,SC,"Abbeville County Sheriff's Office, SC"
4598,Aberdeen,WA,"Aberdeen Police Department, WA"
6474,Abilene,TX,"Abilene Police Department, TX"
5150,Abilene,TX,"Abilene Police Department, TX"
7506,Abilene,TX,"Abilene Police Department, TX"
...,...,...,...
6150,Laredo,TX,"Zapata County Sheriff's Department, TX"
255,Zion,IL,"Zion Police Department, IL"
1010,Zion,IL,"Zion Police Department, IL"
7746,Zion,IL,"Zion Police Department, IL"


In [42]:
def get_mode (data) :
    modes = data.mode()
    if not modes.empty :
        return modes[0]
    else :
        return np.nan

In [43]:
df.groupby('police_departments_involved')['city'].apply(get_mode)

police_departments_involved
Abbeville County Sheriff's Office, SC              Abbeville
Aberdeen Police Department, WA                      Aberdeen
Abilene Police Department, TX                        Abilene
Abington Township Police Department, PA    Abington Township
Acadia Parish Sheriff's Office, LA                   Crowley
                                                 ...        
Yuba County Sheriff's Office, CA                  Olivehurst
Yuma County Sheriff's Department, AZ                    Yuma
Yuma Police Department, AZ                              Yuma
Zapata County Sheriff's Department, TX                Laredo
Zion Police Department, IL                              Zion
Name: city, Length: 3726, dtype: object

In [44]:
dict_of_cites = df.groupby('police_departments_involved')['city'].apply(get_mode).to_dict()

In [45]:
df['city'].fillna(df['police_departments_involved'].map(dict_of_cites) ,  inplace=True)

In [46]:
df['city'].isnull().sum()

38

In [47]:
df[df['city'].isnull()]

,date,name,age,gender,armed,race,city,state,flee,body_camera,signs_of_mental_illness,police_departments_involved,year
2197,2017-03-20,Clarence Duane Huderle,73,male,gun,Unknown,NaN,MN,NaN,False,False,"Polk County Sheriff's Office, MN",2017
8048,2022-12-17,unknown,35,male,gun,Unknown,NaN,WV,NaN,False,False,"McDowell County Sheriff's Office, WV",2022
8054,2022-12-19,Michael Cline,35,male,knife,White,NaN,VA,foot,False,False,"Louisa County Sheriff's Office, VA",2022
8094,2023-01-02,Jerry Preston,49,male,gun,White,NaN,KY,NaN,False,False,"Kentucky State Police, KY;Perry County Sheriff...",2023
8141,2023-01-13,Devin Cribbs,24,male,gun,White,NaN,AL,car,False,False,"Vernon Police Department, AL",2023
8269,2023-02-24,Erin Williamson,28,male,knife,White,NaN,KY,foot,False,False,"Calloway County Sheriff's Department, KY",2023
8372,2023-04-01,Tyler Raymer,28,male,gun,Unknown,NaN,MO,NaN,False,False,"Morgan County Sheriff's Office, MO",2023
8392,2023-04-09,unknown,35,male,gun,Unknown,NaN,MS,NaN,False,False,"Lincoln County Sheriff's Office, MS",2023
8411,2023-04-14,William E. Harver Sr.,45,male,gun,White,NaN,VA,not,False,False,"Amelia County Sheriff's Office, VA",2023
8426,2023-04-20,Dimitri Angel Amarillas,25,male,knife,Hispanic,NaN,TX,NaN,False,False,"Manor Police Department, TX;Travis County Sher...",2023


In [48]:
df[['state', 'city' , 'police_departments_involved']].sort_values(['police_departments_involved' , 'state'])

,state,city,police_departments_involved
6423,SC,Abbeville,"Abbeville County Sheriff's Office, SC"
4598,WA,Aberdeen,"Aberdeen Police Department, WA"
3198,TX,Abilene,"Abilene Police Department, TX"
5150,TX,Abilene,"Abilene Police Department, TX"
6474,TX,Abilene,"Abilene Police Department, TX"
...,...,...,...
6150,TX,Laredo,"Zapata County Sheriff's Department, TX"
255,IL,Zion,"Zion Police Department, IL"
1010,IL,Zion,"Zion Police Department, IL"
7746,IL,Zion,"Zion Police Department, IL"


In [49]:
cities_by_state_mode = df.groupby('state')['city'].transform(get_mode)

In [50]:
df['city'].fillna(cities_by_state_mode , inplace=True)

In [51]:
df['city'].isnull().sum()

0

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9893 entries, 0 to 9892
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         9893 non-null   datetime64[ns]
 1   name                         9893 non-null   object        
 2   age                          9893 non-null   int32         
 3   gender                       9893 non-null   object        
 4   armed                        9893 non-null   object        
 5   race                         9893 non-null   object        
 6   city                         9893 non-null   object        
 7   state                        9893 non-null   object        
 8   flee                         8548 non-null   object        
 9   body_camera                  9893 non-null   bool          
 10  signs_of_mental_illness      9893 non-null   bool          
 11  police_departments_involved  9892 non-null 

In [53]:
from unidecode import unidecode

def cleanCity(CityName):
    standardization = {
        'Ft. ': 'Fort ',
        'Mt. ': 'Mount ',
        'St. ': 'Saint ',
        'Ste. ': 'Sainte ',
        'N. ': 'North ',
        'S. ': 'South ',
        'E. ': 'East ',
        'W. ': 'West ',
        'Lk. ': 'Lake ',
        'Riv. ': 'River ',
        'Crk. ': 'Creek ',
        'Jct. ': 'Junction ',
        'Spgs. ': 'Springs ',
        'Hts. ': 'Heights ',
        'Vly. ': 'Valley '
    }

    suffixes = [
        " unified government (balance)",
        " consolidated government (balance)",
        " city and borough",
        " municipality",
        " village",
        " borough",
        " town",
        " city",
        " CDP",
        "township",
        "Township",
        "county"
    ]

    CityName = str(CityName)

    for abbr, full_word in standardization.items():
        if CityName.startswith(abbr):
            CityName = CityName.replace(abbr, full_word)

    CityName = unidecode(CityName)

    for suffix in suffixes:
        if CityName.endswith(suffix):
            CityName = CityName[:-len(suffix)]
            break

    CityName = CityName.split('(')[0]

    for char in ['.', ',', '"', "'"]:
        CityName = CityName.replace(char, '')

    CityName = ' '.join(CityName.split())
    
    return CityName.strip()


In [54]:
df['city']= df['city'].apply(cleanCity)

# Cleaning State Column

In [55]:
df['state'].nunique()

51

In [56]:
df['state'].unique()

array(['OR', 'WA', 'KS', 'OK', 'CO', 'CA', 'AZ', 'PA', 'IA', 'LA', 'OH',
       'TX', 'UT', 'AR', 'MT', 'NV', 'IL', 'NM', 'NJ', 'VA', 'MO', 'MN',
       'IN', 'KY', 'MA', 'NH', 'FL', 'MD', 'ID', 'NE', 'MI', 'GA', 'TN',
       'NC', 'NY', 'AK', 'ME', 'AL', 'MS', 'WI', 'SC', 'DE', 'DC', 'WV',
       'HI', 'WY', 'ND', 'CT', 'SD', 'VT', 'RI'], dtype=object)

In [57]:
df['state'].isnull().sum()

0

In [58]:
df['state'] =df['state'].str.upper()
df['state']

0       OR
1       WA
2       KS
3       OK
4       CO
        ..
9888    SC
9889    WA
9890    AK
9891    TX
9892    IN
Name: state, Length: 9893, dtype: object

# Cleaning Flee column

In [59]:
df['flee'].isnull().sum()/len(df)*100

13.595471545537249

In [60]:
df['flee'].unique()

array(['not', 'car', 'foot', 'other', nan], dtype=object)

In [61]:
df['flee'].value_counts(normalize=True)

flee
not      0.623304
car      0.182148
foot     0.151614
other    0.042934
Name: proportion, dtype: float64

In [62]:
df.loc[df['flee'].isnull() , 'flee'] = 'Unknown'

In [63]:
df['flee'].value_counts(normalize=True)

flee
not        0.538563
car        0.157384
Unknown    0.135955
foot       0.131002
other      0.037097
Name: proportion, dtype: float64

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9893 entries, 0 to 9892
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         9893 non-null   datetime64[ns]
 1   name                         9893 non-null   object        
 2   age                          9893 non-null   int32         
 3   gender                       9893 non-null   object        
 4   armed                        9893 non-null   object        
 5   race                         9893 non-null   object        
 6   city                         9893 non-null   object        
 7   state                        9893 non-null   object        
 8   flee                         9893 non-null   object        
 9   body_camera                  9893 non-null   bool          
 10  signs_of_mental_illness      9893 non-null   bool          
 11  police_departments_involved  9892 non-null 

# Cleaning bolean columns (body_camera , signs_of_mental_illness)

In [65]:
df['body_camera'].unique()

array([False,  True])

In [66]:
df['signs_of_mental_illness'].unique()

array([False,  True])

# Cleaning police_departments_involved Column

In [67]:
df['police_departments_involved'].isnull().sum()

1

In [68]:
df[df['city'] == 'Lewistown']

,date,name,age,gender,armed,race,city,state,flee,body_camera,signs_of_mental_illness,police_departments_involved,year
2278,2017-04-26,Charles Bossinger,53,male,gun,White,Lewistown,PA,not,False,True,"Lewistown Borough Police Department, PA",2017
5139,2020-03-15,Douglas J. Foster,47,male,gun,White,Lewistown,MT,car,False,False,"Dillon Police Department, MT",2020
5781,2020-11-04,Justin Hammack,26,male,replica,White,Lewistown,IL,Unknown,False,True,"Fulton County Sheriff's Department, IL;Lewisto...",2020
9770,2024-06-10,Ridge Ryen Rhodes,29,male,gun,White,Lewistown,PA,not,False,False,NaN,2024


In [69]:
df.loc[df['police_departments_involved'].isnull() , 'police_departments_involved'] ='Lewistown Borough Police Department, PA'

In [70]:
df['police_departments_involved'].isnull().sum()

0

# Cleaning Diminsions

In [71]:
df2.head()

,year,State,City,Median household income,PercentOfPeopleBelowPovertyLevel,Percent high school graduate or higher,share_white,share_black,share_native,share_asian,share_hispanic
0,2015,Alabama,Abanda CDP,11207,78.8,21.2,76.9,23.1,0.0,0.0,0.0
1,2015,Alabama,Abbeville city,25615,29.1,69.1,52.8,46.4,0.0,0.0,1.0
2,2015,Alabama,Adamsville city,42575,25.5,78.9,47.6,43.3,0.0,0.5,9.1
3,2015,Alabama,Addison town,37083,30.7,81.4,94.0,0.0,0.3,0.0,0.0
4,2015,Alabama,Akron town,21667,42.0,68.6,20.1,79.9,0.0,0.0,0.0


In [72]:
States.head()

,Id,StateName,Code
0,1,Alabama,AL
1,2,Alaska,AK
2,3,Arizona,AZ
3,4,Arkansas,AR
4,5,California,CA


In [73]:
States.drop('Id' , axis=1 , inplace=True)

In [74]:
States.rename(columns={'StateName' : 'State'} , inplace=True)

In [75]:
df2['State']=df2['State'].str.lower()
df2['State']=df2['State'].str.strip()
df2['State']=df2['State'].str.capitalize()

States['State'] =States['State'].str.lower()
States['State'] =States['State'].str.strip()
States['State'] =States['State'].str.capitalize()

In [76]:
df2 = pd.merge(df2 , States , how='left' )

In [77]:
len(df2)

276197

In [78]:
df.head()

,date,name,age,gender,armed,race,city,state,flee,body_camera,signs_of_mental_illness,police_departments_involved,year
0,2015-01-02,Lewis Lee Lembke,47,male,gun,White,Aloha,OR,not,False,False,"Washington County Sheriff's Office, OR",2015
1,2015-01-02,Tim Elliot,53,male,gun,Asian,Shelton,WA,not,False,True,"Mason County Sheriff's Office, WA",2015
2,2015-01-03,John Paul Quintero,23,male,unarmed,Hispanic,Wichita,KS,not,False,False,"Wichita Police Department, KS",2015
3,2015-01-04,Kenneth Joe Brown,18,male,gun,White,Guthrie,OK,not,False,False,"Oklahoma Highway Patrol, OK",2015
4,2015-01-04,Michael Rodriguez,39,male,other,Hispanic,Evans,CO,not,False,False,"Evans Police Department, CO",2015


In [79]:
df.rename(columns={'state' : 'Code'} , inplace=True)

In [80]:
df['city']= df['city'].apply(cleanCity)

In [81]:
df2['City']= df2['City'].apply(cleanCity)

In [82]:
def normalize_city(series):
    series = series.astype(str).str.lower().str.strip()
    series = series.str.replace(r'[^\w\s]', '', regex=True)  # remove punctuation
    
    # remove common suffixes
    series = series.str.replace(r'\bcounty\b', '', regex=True)
    series = series.str.replace(r'\bparish\b', '', regex=True)
    series = series.str.replace(r'\bcity\b', '', regex=True)
    series = series.str.replace(r'\bvillage\b', '', regex=True)
    series = series.str.replace(r'\btownship\b', '', regex=True)

    # standardize common abbreviations
    series = series.str.replace(r'\bst\b', 'saint', regex=True)
    series = series.str.replace(r'\bft\b', 'fort', regex=True)
    series = series.str.replace(r'\bn\b', 'north', regex=True)
    series = series.str.replace(r'\bs\b', 'south', regex=True)
    series = series.str.replace(r'\bw\b', 'west', regex=True)
    series = series.str.replace(r'\be\b', 'east', regex=True)

    # clean spaces
    series = series.str.replace(r'\s+', ' ', regex=True).str.strip()
    return series


def find_best_match_by_state(messy_name, state, choices_map, all_choices, score_cutoff=75):
    # first try state-specific
    if state in choices_map and choices_map[state]:
        best_match = process.extractOne(
            messy_name,
            choices_map[state],
            scorer=fuzz.token_set_ratio
        )
        if best_match and best_match[1] >= score_cutoff:
            return best_match[0]

    # fallback: try all cities
    best_match = process.extractOne(
        messy_name,
        all_choices,
        scorer=fuzz.token_set_ratio
    )
    if best_match and best_match[1] >= score_cutoff:
        return best_match[0]

    return None


# --- cleaning ---
df['clean_city'] = normalize_city(df['city'])
df['clean_state'] = normalize_city(df['Code'])

df2['clean_city'] = normalize_city(df2['City'])
df2['clean_state'] = normalize_city(df2['Code'])

# --- choices ---
choices_by_state = df.groupby('clean_state')['clean_city'].apply(list).to_dict()
all_choices = df['clean_city'].unique().tolist()

# --- apply matching ---
df2['Corrected_City'] = df2.apply(
    lambda row: find_best_match_by_state(
        row['clean_city'], 
        row['clean_state'], 
        choices_by_state, 
        all_choices
    ),
    axis=1
)

# fill with original if no match
df2['Corrected_City'].fillna(df2['clean_city'], inplace=True)

# update City column
df2['City'] = df2['Corrected_City']

# drop helper columns
df2.drop(columns=['clean_city', 'clean_state', 'Corrected_City'], inplace=True)
# df.drop(columns=['clean_city', 'clean_state'], inplace=True)

# --- export unmatched for manual review ---
unmatched = df2[df2['City'].isna()]
if not unmatched.empty:
    unmatched.to_csv("unmatched_cities.csv", index=False)

print(df2.head())

   year    State       City Median household income  \
0  2015  Alabama    bandera                   11207   
1  2015  Alabama  abbeville                   25615   
2  2015  Alabama   ashville                   42575   
3  2015  Alabama    madison                   37083   
4  2015  Alabama      akron                   21667   

  PercentOfPeopleBelowPovertyLevel Percent high school graduate or higher  \
0                             78.8                                   21.2   
1                             29.1                                   69.1   
2                             25.5                                   78.9   
3                             30.7                                   81.4   
4                             42.0                                   68.6   

   share_white  share_black  share_native  share_asian  share_hispanic Code  
0         76.9         23.1           0.0          0.0             0.0   AL  
1         52.8         46.4           0.0         

In [83]:
city1 = set(df['clean_city'])
city2 = set(df2['City'])

In [84]:
defra = city1.difference(city2)

In [85]:
len(defra)

134

In [86]:
df['city'] = df['clean_city']

In [87]:
df.drop(columns=['clean_city', 'clean_state'], inplace=True)

In [90]:
df2 = df2.sort_values(by=['State', 'City', 'year'])

In [93]:
df2

,year,State,City,Median household income,PercentOfPeopleBelowPovertyLevel,Percent high school graduate or higher,share_white,share_black,share_native,share_asian,share_hispanic,Code
1,2015,Alabama,abbeville,25615.0,29.1,69.1,52.8,46.4,0.0,0.0,1.0,AL
29575,2016,Alabama,abbeville,28148.0,24.6,77.5,52.8,46.1,0.0,0.0,1.5,AL
59149,2017,Alabama,abbeville,40724.0,20.7,79.1,56.4,41.8,0.0,0.0,4.5,AL
88725,2018,Alabama,abbeville,42877.0,15.8,79.5,60.0,39.7,0.0,0.0,4.8,AL
118298,2019,Alabama,abbeville,36875.0,24.2,77.8,54.7,41.5,0.9,0.0,5.5,AL
...,...,...,...,...,...,...,...,...,...,...,...,...
147615,2019,Wyoming,yoder,38229.0,4.1,88.3,100.0,0.0,0.0,0.0,1.2,WY
179484,2020,Wyoming,yoder,38125.0,16.9,88.4,94.0,0.0,0.0,0.0,0.0,WY
211392,2021,Wyoming,yoder,32917.0,20.9,88.7,93.6,0.0,0.0,0.0,0.0,WY
243578,2022,Wyoming,yoder,27417.0,30.4,89.4,90.2,0.0,0.0,0.0,0.0,WY


In [94]:
columns_to_process = [
    'Median household income',
    'PercentOfPeopleBelowPovertyLevel',
    'Percent high school graduate or higher',
    'share_white',
    'share_black',
    'share_native',
    'share_asian',
    'share_hispanic'
]

In [95]:
#change columns to numric
for col in columns_to_process:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

In [96]:
#display nulls precntage
for col in columns_to_process:
    print(col , " = " ,round(df2[col].isnull().sum()/len(df2[col])*100 , 2 ), "%")

Median household income  =  9.1 %
PercentOfPeopleBelowPovertyLevel  =  1.24 %
Percent high school graduate or higher  =  1.1 %
share_white  =  1.19 %
share_black  =  1.19 %
share_native  =  1.19 %
share_asian  =  1.19 %
share_hispanic  =  1.19 %


In [97]:
#after sorting the data as before 
# For each column, fill missing values by interpolating within each unique State-City group.
for col in columns_to_process:
    df2[col] = df2.groupby(['State', 'City'])[col].transform(lambda x: x.interpolate(limit_direction='both'))


In [98]:
#display nulls precntage
for col in columns_to_process:
    print(col , " = " ,round(df2[col].isnull().sum()/len(df2[col])*100 , 2 ), "%")

Median household income  =  2.28 %
PercentOfPeopleBelowPovertyLevel  =  0.28 %
Percent high school graduate or higher  =  0.26 %
share_white  =  0.34 %
share_black  =  0.34 %
share_native  =  0.34 %
share_asian  =  0.34 %
share_hispanic  =  0.34 %


In [100]:
# Fill the remaining nulls with the median of each state for each year individually
for col in columns_to_process:
    median_fill_values = df2.groupby(['State', 'year'])[col].transform('median')
    df2[col].fillna(median_fill_values, inplace=True)

In [101]:
#display nulls precntage
for col in columns_to_process:
    print(col , " = " ,round(df2[col].isnull().sum()/len(df2[col])*100 , 2 ), "%")

Median household income  =  0.0 %
PercentOfPeopleBelowPovertyLevel  =  0.0 %
Percent high school graduate or higher  =  0.0 %
share_white  =  0.0 %
share_black  =  0.0 %
share_native  =  0.0 %
share_asian  =  0.0 %
share_hispanic  =  0.0 %


In [102]:
df2

,year,State,City,Median household income,PercentOfPeopleBelowPovertyLevel,Percent high school graduate or higher,share_white,share_black,share_native,share_asian,share_hispanic,Code
1,2015,Alabama,abbeville,25615.0,29.1,69.1,52.8,46.4,0.0,0.0,1.0,AL
29575,2016,Alabama,abbeville,28148.0,24.6,77.5,52.8,46.1,0.0,0.0,1.5,AL
59149,2017,Alabama,abbeville,40724.0,20.7,79.1,56.4,41.8,0.0,0.0,4.5,AL
88725,2018,Alabama,abbeville,42877.0,15.8,79.5,60.0,39.7,0.0,0.0,4.8,AL
118298,2019,Alabama,abbeville,36875.0,24.2,77.8,54.7,41.5,0.9,0.0,5.5,AL
...,...,...,...,...,...,...,...,...,...,...,...,...
147615,2019,Wyoming,yoder,38229.0,4.1,88.3,100.0,0.0,0.0,0.0,1.2,WY
179484,2020,Wyoming,yoder,38125.0,16.9,88.4,94.0,0.0,0.0,0.0,0.0,WY
211392,2021,Wyoming,yoder,32917.0,20.9,88.7,93.6,0.0,0.0,0.0,0.0,WY
243578,2022,Wyoming,yoder,27417.0,30.4,89.4,90.2,0.0,0.0,0.0,0.0,WY


In [103]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 276197 entries, 1 to 275903
Data columns (total 12 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   year                                    276197 non-null  int64  
 1   State                                   276197 non-null  object 
 2   City                                    276197 non-null  object 
 3   Median household income                 276197 non-null  float64
 4   PercentOfPeopleBelowPovertyLevel        276197 non-null  float64
 5   Percent high school graduate or higher  276197 non-null  float64
 6   share_white                             276197 non-null  float64
 7   share_black                             276197 non-null  float64
 8   share_native                            276197 non-null  float64
 9   share_asian                             276197 non-null  float64
 10  share_hispanic                          276197 no

In [107]:
columns_to_check = [
    'PercentOfPeopleBelowPovertyLevel',
    'Percent high school graduate or higher',
    'share_white',
    'share_black',
    'share_native',
    'share_asian',
    'share_hispanic'
]

for col in columns_to_check:
    outliers = df2[(df2[col] < 0) | (df2[col] > 100)]
    if not outliers.empty:
        print(f" there is wrong value in :  '{col}':")
        print(outliers[[col]]) 
    else:
        print(f" column '{col}'is clean no problems" )

print("Chek is done")

 column 'PercentOfPeopleBelowPovertyLevel' is clean no problems
 column 'Percent high school graduate or higher' is clean no problems
 column 'share_white' is clean no problems
 column 'share_black' is clean no problems
 column 'share_native' is clean no problems
 column 'share_asian' is clean no problems
 column 'share_hispanic' is clean no problems
Chek is done


In [108]:
df2.isnull().sum()

year                                      0
State                                     0
City                                      0
Median household income                   0
PercentOfPeopleBelowPovertyLevel          0
Percent high school graduate or higher    0
share_white                               0
share_black                               0
share_native                              0
share_asian                               0
share_hispanic                            0
Code                                      0
dtype: int64

In [167]:
df.drop_duplicates(inplace=True)

In [168]:
df2.drop_duplicates(subset=['City', 'Code', 'year'] , inplace=True)

In [169]:
df.rename(columns={'city' : 'City'} , inplace=True)

In [170]:
Data = pd.merge(df , df2 , on=['City' , 'Code' , 'year'] , how='left').drop_duplicates()

In [171]:
Data

,date,name,age,gender,armed,race,City,Code,flee,body_camera,...,year,State,Median household income,PercentOfPeopleBelowPovertyLevel,Percent high school graduate or higher,share_white,share_black,share_native,share_asian,share_hispanic
0,2015-01-02,Lewis Lee Lembke,47,male,gun,White,aloha,OR,not,False,...,2015,Oregon,65765.0,14.9,88.1,70.7,4.0,1.0,8.6,23.5
1,2015-01-02,Tim Elliot,53,male,gun,Asian,shelton,WA,not,False,...,2015,Washington,37072.0,28.6,80.1,76.0,0.5,2.9,1.0,21.3
2,2015-01-03,John Paul Quintero,23,male,unarmed,Hispanic,wichita,KS,not,False,...,2015,Kansas,45947.0,17.3,87.5,75.6,11.5,1.0,5.1,16.0
3,2015-01-04,Kenneth Joe Brown,18,male,gun,White,guthrie,OK,not,False,...,2015,Oklahoma,41267.0,18.6,89.7,72.6,18.7,4.6,0.4,3.9
4,2015-01-04,Michael Rodriguez,39,male,other,Hispanic,evans,CO,not,False,...,2015,Colorado,47791.0,16.6,76.3,86.3,0.2,1.4,1.7,47.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9888,2024-07-13,Daniel Scott McGoldrick,35,male,gun,Unknown,easley,SC,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9889,2024-07-15,unknown,35,male,vehicle,Unknown,graham,WA,other,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9890,2024-07-15,Steven Kissack,35,male,knife,White,juneau,AK,foot,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9891,2024-07-15,Charles Patrick Carroll,68,male,replica,White,beaumont,TX,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [172]:
Data.to_csv('D:\CRIME DATA SET\DATA\Project\CleanDataSet.csv')

In [173]:
df.to_csv('D:\CRIME DATA SET\DATA\Project\CleanFactTable.csv')

In [174]:
df2.to_csv('D:\CRIME DATA SET\DATA\Project\CleanDimTables.csv')

In [159]:
mask = Data['year'] == 2024

In [160]:
Data24 = Data[mask]

In [163]:
Data24

,date,name,age,gender,armed,race,City,Code,flee,body_camera,...,year,State,Median household income,PercentOfPeopleBelowPovertyLevel,Percent high school graduate or higher,share_white,share_black,share_native,share_asian,share_hispanic
9252,2024-01-01,Sidney Tafokitau,44,male,gun,Other,honolulu,HI,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9253,2024-01-01,Katelynn Rose Smith,29,female,gun,Unknown,longview,WA,not,True,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9254,2024-01-01,Victor Figueroa Roblero,48,male,unarmed,Hispanic,spartanburg,SC,other,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9255,2024-01-01,Aaron Travis Watson,35,male,gun,White,little rock,AR,car,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9256,2024-01-03,Rakim A. Tillery,35,male,gun,Black,ramapo,NY,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9888,2024-07-13,Daniel Scott McGoldrick,35,male,gun,Unknown,easley,SC,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9889,2024-07-15,unknown,35,male,vehicle,Unknown,graham,WA,other,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9890,2024-07-15,Steven Kissack,35,male,knife,White,juneau,AK,foot,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9891,2024-07-15,Charles Patrick Carroll,68,male,replica,White,beaumont,TX,not,False,...,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [164]:
Data.isnull().sum()

date                                        0
name                                        0
age                                         0
gender                                      0
armed                                       0
race                                        0
City                                        0
Code                                        0
flee                                        0
body_camera                                 0
signs_of_mental_illness                     0
police_departments_involved                 0
year                                        0
State                                     825
Median household income                   825
PercentOfPeopleBelowPovertyLevel          825
Percent high school graduate or higher    825
share_white                               825
share_black                               825
share_native                              825
share_asian                               825
share_hispanic                    

In [ ]:
Data